# 05 - Hemisphere Analysis
Compare left vs right hemisphere performance and reorganization patterns.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import json

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

In [ ]:
# Setup
PROJECT_ROOT = Path.cwd().parent if 'analysis' in str(Path.cwd()) else Path.cwd()
RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'
ANALYSIS_DIR = PROJECT_ROOT / 'data' / 'analysis'
FIGURES_DIR = PROJECT_ROOT / 'figures' / 'hemisphere'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Hemisphere Results

In [ ]:
# Load compiled results
cv_perf = pd.read_csv(ANALYSIS_DIR / 'compiled_cv_performance.csv')
task_perf = pd.read_csv(ANALYSIS_DIR / 'compiled_task_performance.csv')

# Filter hemisphere results
left_cv = cv_perf[cv_perf['Analysis'] == 'Left Hemisphere'].copy()
right_cv = cv_perf[cv_perf['Analysis'] == 'Right Hemisphere'].copy()
left_task = task_perf[task_perf['Analysis'] == 'Left Hemisphere'].copy()
right_task = task_perf[task_perf['Analysis'] == 'Right Hemisphere'].copy()

print(f"Left CV: {len(left_cv)}, Right CV: {len(right_cv)}")
print(f"Left Task: {len(left_task)}, Right Task: {len(right_task)}")

## 2. Compare Performance

In [ ]:
# Compare CV performance
print("Cross-Validation Performance:")
print("\nLeft Hemisphere:")
print(left_cv[['Strategy', 'Accuracy', 'Balanced_Accuracy', 'Macro_F1']].to_string(index=False))
print("\nRight Hemisphere:")
print(right_cv[['Strategy', 'Accuracy', 'Balanced_Accuracy', 'Macro_F1']].to_string(index=False))

In [ ]:
# Compare task performance
print("\nTask Testing Performance:")
print("\nLeft Hemisphere:")
print(left_task[['Strategy', 'Accuracy', 'Balanced_Accuracy', 'Macro_F1']].to_string(index=False))
print("\nRight Hemisphere:")
print(right_task[['Strategy', 'Accuracy', 'Balanced_Accuracy', 'Macro_F1']].to_string(index=False))

In [ ]:
# Plot comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

metrics = ['Accuracy', 'Balanced_Accuracy', 'Macro_F1', 'Weighted_F1']
titles = ['Accuracy', 'Balanced Accuracy', 'Macro F1', 'Weighted F1']

for idx, (metric, title) in enumerate(zip(metrics, titles)):
    ax = axes[idx // 2, idx % 2]
    
    # Prepare data
    strategies = left_cv['Strategy'].values
    left_vals = left_cv[metric].values
    right_vals = right_cv[metric].values
    
    x = np.arange(len(strategies))
    width = 0.35
    
    ax.bar(x - width/2, left_vals, width, label='Left', color='steelblue', edgecolor='black')
    ax.bar(x + width/2, right_vals, width, label='Right', color='coral', edgecolor='black')
    
    ax.set_xlabel('Strategy', fontsize=11)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(f'CV {title}: Left vs Right Hemisphere', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(strategies)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'hemisphere_cv_comparison.png', dpi=300, bbox_inches='tight')
print("\nSaved hemisphere CV comparison")
plt.show()

## 3. Statistical Tests

In [ ]:
# Paired t-tests
print("\nPaired t-tests (Left vs Right):")
for metric in ['Accuracy', 'Balanced_Accuracy', 'Macro_F1']:
    t_stat, p_val = stats.ttest_rel(left_cv[metric], right_cv[metric])
    print(f"\n{metric}:")
    print(f"  t-statistic: {t_stat:.4f}")
    print(f"  p-value: {p_val:.4f}")
    print(f"  Significant: {'Yes' if p_val < 0.05 else 'No'}")

## 4. Save Results

In [ ]:
# Create hemisphere comparison summary
hemisphere_summary = pd.DataFrame({
    'Metric': ['Accuracy', 'Balanced_Accuracy', 'Macro_F1'],
    'Left_Mean': [left_cv['Accuracy'].mean(), left_cv['Balanced_Accuracy'].mean(), left_cv['Macro_F1'].mean()],
    'Right_Mean': [right_cv['Accuracy'].mean(), right_cv['Balanced_Accuracy'].mean(), right_cv['Macro_F1'].mean()],
    'Difference': [left_cv['Accuracy'].mean() - right_cv['Accuracy'].mean(),
                  left_cv['Balanced_Accuracy'].mean() - right_cv['Balanced_Accuracy'].mean(),
                  left_cv['Macro_F1'].mean() - right_cv['Macro_F1'].mean()]
})

hemisphere_summary.to_csv(ANALYSIS_DIR / 'hemisphere_comparison.csv', index=False)
print("\n" + "="*80)
print("HEMISPHERE ANALYSIS COMPLETE")
print("="*80)